In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import duckdb

In [2]:
shots = pd.read_csv('../scraper/mens/shots.csv')
shots['scoringPlay'] = shots['scoringPlay'].notna() #make scoring T/F
shots = shots.rename(columns={'athlete.id':'athleteId'})
players = pd.read_csv('../scraper/mens/players.csv')

shots = shots[shots['athleteId'].isin(players['id'])].reset_index(drop=True) #get Cornell players only
shots = shots[['id','text','scoringPlay','pointsAttempted','athleteId','type.categoryId','type.txt','gameId']]
non_ft_shots = shots[shots['type.txt'] != 'MadeFreeThrow'] # made free throw includes misses, can use != 1001 or 1004 for typeID too

### Work for Table 1 (Probability of Making a Shot Conditioned on the Outcome of Previous Shots)

In [3]:
shots_conditional = duckdb.sql("""SELECT *, 
                  LAG(scoringPlay, 1, NULL) OVER(PARTITION BY athleteId, gameId ORDER BY id) AS prevShot,
                  LAG(scoringPlay, 2, NULL) OVER(PARTITION BY athleteId, gameId ORDER BY id) AS prev2Shot,
                  LAG(scoringPlay, 3, NULL) OVER(PARTITION BY athleteId, gameId ORDER BY id) AS prev3Shot,
                  FROM non_ft_shots
                  ORDER BY athleteId, id""").df()

shots_conditional.head()

,id,text,scoringPlay,pointsAttempted,athleteId,type.categoryId,type.txt,gameId,prevShot,prev2Shot,prev3Shot
0,401809079115260304,DJ Nix missed Layup.,False,2,4684840,1005,LayUpShot,401809079,<NA>,<NA>,<NA>
1,401809079115263385,DJ Nix missed Three Point Jumper.,False,3,4684840,1006,JumpShot,401809079,False,<NA>,<NA>
2,401809079115270616,DJ Nix missed Three Point Jumper.,False,3,4684840,1006,JumpShot,401809079,False,False,<NA>
3,401809079115273445,DJ Nix made Layup. Assisted by Kaspar Sepp.,True,2,4684840,1002,LayUpShot,401809079,False,False,False
4,401809079115275457,DJ Nix made Three Point Jumper. Assisted by Ad...,True,3,4684840,1003,JumpShot,401809079,True,False,False


In [4]:
p_hit_3miss = duckdb.sql("""SELECT athleteId, COUNT() as count3Misses, AVG(CASE WHEN scoringPlay = TRUE THEN 1 ELSE 0 END) as prHit3Misses
                         FROM shots_conditional
                         WHERE prevShot = False AND prev2Shot = False AND prev3Shot = False
                         GROUP BY athleteId""").df()

p_hit_2miss = duckdb.sql("""SELECT athleteId, COUNT() as count2Misses, AVG(CASE WHEN scoringPlay = TRUE THEN 1 ELSE 0 END) as prHit2Misses 
                         FROM shots_conditional
                         WHERE prevShot = False AND prev2Shot = False
                         GROUP BY athleteId""").df()

p_hit_miss = duckdb.sql("""SELECT athleteId, COUNT() as count1Miss, AVG(CASE WHEN scoringPlay = TRUE THEN 1 ELSE 0 END) as prHit1Miss 
                         FROM shots_conditional
                         WHERE prevShot = False
                         GROUP BY athleteId""").df()

p_hit_3makes = duckdb.sql("""SELECT athleteId, COUNT() as count3Makes, AVG(CASE WHEN scoringPlay = TRUE THEN 1 ELSE 0 END) as prHit3Makes
                         FROM shots_conditional
                         WHERE prevShot = True AND prev2Shot = True AND prev3Shot = True
                         GROUP BY athleteId""").df()

p_hit_2makes = duckdb.sql("""SELECT athleteId, COUNT() as count2Makes, AVG(CASE WHEN scoringPlay = TRUE THEN 1 ELSE 0 END) as prHit2Makes 
                         FROM shots_conditional
                         WHERE prevShot = True AND prev2Shot = True
                         GROUP BY athleteId""").df()

p_hit_make = duckdb.sql("""SELECT athleteId, COUNT() as count1Make, AVG(CASE WHEN scoringPlay = TRUE THEN 1 ELSE 0 END) as prHit1Make
                         FROM shots_conditional
                         WHERE prevShot = True
                         GROUP BY athleteId""").df()

p_hit = duckdb.sql("""SELECT athleteId, COUNT() as countHit, AVG(CASE WHEN scoringPlay = TRUE THEN 1 ELSE 0 END) as prHit
                         FROM shots_conditional
                         GROUP BY athleteId""").df()

In [5]:
table = duckdb.sql("""SELECT * FROM
                    p_hit
                    LEFT JOIN p_hit_3miss ON p_hit.athleteId = p_hit_3miss.athleteId
                    LEFT JOIN p_hit_2miss ON p_hit.athleteId = p_hit_2miss.athleteId
                    LEFT JOIN p_hit_miss ON p_hit.athleteId = p_hit_miss.athleteId
                    LEFT JOIN p_hit_make ON p_hit.athleteId = p_hit_make.athleteId
                    LEFT JOIN p_hit_2makes ON p_hit.athleteId = p_hit_2makes.athleteId
                    LEFT JOIN p_hit_3makes ON p_hit.athleteId = p_hit_3makes.athleteId""").df()

table = table.loc[:,~table.columns.str.startswith('athleteId_')] #removes duplicate athleteId columns
table.to_csv('mens_table1.csv', index=False)